# Step 7: Error Analysis
Pull the 30 validation examples the model was most confident about but got wrong.


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification


In [ ]:
# Load validation data and best model
val_df = pd.read_csv("data/processed/validation.csv").dropna(subset=['text'])
model_path = "./best_model"

try:
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
except Exception as e:
    print("Could not load model. Ensure Step 6 completed and saved the model to ./best_model")
    raise

model.eval()
if torch.cuda.is_available():
    model.to('cuda')


In [ ]:
# Predict and get probabilities
texts = val_df['text'].tolist()

all_probs = []
all_preds = []

batch_size = 32
print("Running inference on validation set...")
for i in range(0, len(texts), batch_size):
    batch_texts = texts[i:i+batch_size]
    inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
    
    if torch.cuda.is_available():
        inputs = {k: v.to('cuda') for k, v in inputs.items()}
        
    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1)
        preds = torch.argmax(probs, dim=-1)
        
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

val_df['predicted_label'] = all_preds
val_df['prob_0'] = [p[0] for p in all_probs]
val_df['prob_1'] = [p[1] for p in all_probs]


In [ ]:
# Find confident errors
errors = val_df[val_df['label'] != val_df['predicted_label']].copy()
errors['confidence'] = errors[['prob_0', 'prob_1']].max(axis=1)

# Top 30 most confident errors
top_errors = errors.sort_values(by='confidence', ascending=False).head(30)
print(f"Found {len(errors)} total errors.")
print("\nTop 30 Most Confident Errors:")
display(top_errors[['text', 'label', 'predicted_label', 'confidence']])


### Categorization Task
Please manually categorize these 30 examples into:
1. **Label Noise** (the true label is actually wrong)
2. **Ambiguous Content** (hard to tell even for a human)
3. **Register/Style Overfitting** (e.g. looks like a real news article but is fake)
4. **Genuinely hard cases**
